In [98]:
model = cp_model.CpModel()
solver = cp_model.CpSolver()

In [99]:
courses = [\
    {"id": 1, "name": "Analiza"},\
    {"id": 3, "name": "ORS"},\
    {"id": 1, "name": "Analiza"},\
    {"id": 1, "name": "Analiza"},\
    {"id": 2, "name": "Algebra"},\
    {"id": 1, "name": "Analiza"},\
    {"id": 2, "name": "Algebra"},\
    {"id": 3, "name": "ORS"},\
    {"id": 2, "name": "Algebra"}\
]

In [100]:
n = len(courses)
total_slots = 10
variables = []
for i in range(n):
    variables.append(model.NewIntVar(0, total_slots, f"var_{i}"))
model.AddAllDifferent(variables)

In [101]:
def print_timetable(courses, variables):
    for i in range(n):
        print(f"course {courses[i]['name']} starts at {solver.Value(variables[i])}")

In [102]:
grouped_courses = defaultdict(list)
for idx, course in enumerate(courses):
    grouped_courses[course["id"]].append(idx)

In [103]:
for course_id, course_idx in grouped_courses.items():
    block_sizes = [2,2] if len(course_idx) > 3 else [len(course_idx)]

    offset = 0
    for block_size in block_sizes:
        block = course_idx[offset : offset + block_size]        
        offset += block_size

        anchor = block[0]

        for i, c in enumerate(block[1:]):            
            model.Add(variables[c] == variables[anchor] + i + 1)

In [112]:
time_slots_busy = []
for t in range(total_slots):
    lits = []
    for c_idx, course in enumerate(courses):
        lit = model.NewBoolVar(f"c_{c_idx}_{t}");
        model.Add(variables[c_idx] == t).OnlyEnforceIf(lit)
        model.Add(variables[c_idx] != t).OnlyEnforceIf(lit.Not())
        lits.append(lit)
        
    b = model.NewBoolVar(f"busy_{t}")
    model.AddMaxEquality(b, lits)
    time_slots_busy.append(b)

gaps = []
for t in range(1, total_slots-1):
    has_before = model.NewBoolVar(f"before_{t}")
    model.AddMaxEquality(has_before, time_slots_busy[:t])
    has_after = model.NewBoolVar(f"after_{t}")
    model.AddMaxEquality(has_after, time_slots_busy[t+1:])

    gap = model.NewBoolVar(f"gap_hour_{t}")
    model.AddBoolAnd([has_before, has_after, time_slots_busy[t].Not()]).OnlyEnforceIf(gap)
    model.AddBoolOr([has_before.Not(), has_after.Not(), time_slots_busy[t]]).OnlyEnforceIf(gap.Not())

    gaps.append(gap)

for gap in gaps:
    model.Add(gap == 0)

In [113]:
solver.Solve(model)

<CpSolverStatus.OPTIMAL: 4>

In [117]:
print_timetable(courses, variables)

course Analiza starts at 7
course ORS starts at 3
course Analiza starts at 8
course Analiza starts at 5
course Algebra starts at 0
course Analiza starts at 6
course Algebra starts at 1
course ORS starts at 4
course Algebra starts at 2


In [115]:
for time_slot in time_slots_busy:
    print(solver.Value(time_slot))

1
1
1
1
1
1
1
1
1
0


In [116]:
for gap in gaps:
    print(solver.Value(gap))

0
0
0
0
0
0
0
0
